# SceneSense - Image Classification

SceneSense is an image classification system that classifies natural scene images into 6 categories using Convolutional Neural Networks (CNN) with TensorFlow.

**Classes:** buildings, forest, glacier, mountain, sea, street

## 1. Import

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('.'))

import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

from src.config import Config
from src.utils import set_seed, plot_history, get_timestamp
from src.dataset import SceneSenseDataset
from src.augmentation import AugmentationPipeline
from src.model import SceneSenseModel
from src.trainer import Trainer
from src.evaluator import Evaluator
from src.exporter import Exporter

print("All imports successful.")

## 2. Configuration

In [ ]:
config = Config()
set_seed(config.RANDOM_SEED)
timestamp = get_timestamp()

print(f"Image Size: {config.IMAGE_SIZE}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Epochs: {config.EPOCHS}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Classes: {config.CLASSES}")

## 3. Dataset Exploration

In [ ]:
dataset = SceneSenseDataset(config)

if not os.path.isdir(os.path.join(config.DATASET_PATH, 'train')):
    print("Dataset not split yet. Use --data_dir with train.py or place raw data.")
else:
    dataset.load_datasets()
    stats = dataset.get_dataset_statistics()
    print("Dataset Statistics:")
    for key, val in stats.items():
        print(f"  {key}: {val}")
    print(f"\nClass names: {dataset.class_names}")

In [ ]:
if dataset.train_ds:
    plt.figure(figsize=(12, 8))
    for images, labels in dataset.train_ds.take(1):
        for i in range(9):
            plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].numpy().astype("uint8"))
            plt.title(dataset.class_names[labels[i]])
            plt.axis("off")
    plt.tight_layout()
    plt.show()

## 4. Data Split

The dataset has been split into **Train (70%)**, **Validation (15%)**, and **Test (15%)** sets using stratified splitting to maintain class distribution.

In [ ]:
if dataset.train_ds:
    train_count = sum(1 for _ in dataset.train_ds.unbatch())
    val_count = sum(1 for _ in dataset.val_ds.unbatch())
    test_count = sum(1 for _ in dataset.test_ds.unbatch())
    print(f"Train samples: {train_count}")
    print(f"Validation samples: {val_count}")
    print(f"Test samples: {test_count}")
    print(f"Total: {train_count + val_count + test_count}")

## 5. Data Augmentation

Augmentation (RandomFlip, RandomRotation, RandomZoom, RandomTranslation, RandomBrightness, RandomContrast) is applied only to the **training** dataset to improve generalization.

In [ ]:
if dataset.train_ds:
    aug_pipeline = AugmentationPipeline(config)
    train_ds = dataset.train_ds.map(aug_pipeline.apply, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
    val_ds = dataset.val_ds.prefetch(tf.data.AUTOTUNE)
    test_ds = dataset.test_ds.prefetch(tf.data.AUTOTUNE)
    print("Augmentation pipeline ready.")
    
    plt.figure(figsize=(12, 8))
    for images, labels in dataset.train_ds.take(1):
        aug_images, _ = aug_pipeline.apply(images, labels)
        for i in range(9):
            plt.subplot(3, 3, i + 1)
            plt.imshow(aug_images[i].numpy())
            plt.title(f"Augmented: {dataset.class_names[labels[i]]}")
            plt.axis("off")
    plt.tight_layout()
    plt.show()

## 6. Model

Using TensorFlow **Sequential API** with 4 dual-conv blocks (32,64,128,256 filters) + GlobalAveragePooling2D + Dense(512) + Dropout.

In [ ]:
model_builder = SceneSenseModel(config)
model = model_builder.get_model()
model.summary()

## 7. Training

Training with EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TerminateOnNaN, and CSVLogger callbacks.

In [ ]:
if dataset.train_ds:
    trainer = Trainer(config)
    history = trainer.train(model, train_ds, val_ds)
    print("Training completed.")

## 8. Evaluation

Evaluating on the **test set** with accuracy, loss, confusion matrix, and classification report.

In [ ]:
if dataset.train_ds:
    evaluator = Evaluator(config)
    results = evaluator.evaluate(model, test_ds, dataset.class_names)
    print(f"\nTest Accuracy: {results['accuracy']:.4f}")
    print(f"Test Loss: {results['loss']:.4f}")

In [ ]:
if dataset.train_ds:
    plot_history(history, output_path=config.OUTPUT_PATH, timestamp=timestamp)
    from IPython.display import Image as IPImage
    display(IPImage(os.path.join(config.OUTPUT_PATH, f"accuracy_{timestamp}.png")))
    display(IPImage(os.path.join(config.OUTPUT_PATH, f"loss_{timestamp}.png")))

## 9. Inference Example

In [ ]:
if dataset.train_ds:
    test_dir = os.path.join(config.DATASET_PATH, "test")
    example_images = []
    for cls_name in dataset.class_names:
        cls_dir = os.path.join(test_dir, cls_name)
        if os.path.isdir(cls_dir):
            files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if files:
                example_images.append(os.path.join(cls_dir, files[0]))
    
    if example_images:
        for img_path in example_images[:3]:
            from src.utils import load_image_for_inference
            from PIL import Image
            img = load_image_for_inference(img_path, target_size=config.IMAGE_SIZE)
            preds = model.predict(img, verbose=0)[0]
            predicted_idx = int(np.argmax(preds))
            confidence = float(preds[predicted_idx])
            
            plt.figure(figsize=(4, 3))
            plt.imshow(Image.open(img_path).resize(config.IMAGE_SIZE))
            plt.title(f"Pred: {dataset.class_names[predicted_idx]} ({confidence:.2%})")
            plt.axis("off")
            plt.show()

## 10. Export

Exporting model to **SavedModel**, **TensorFlow Lite**, and **TensorFlow.js** formats.

In [ ]:
exporter = Exporter(config)
exporter.export_all(model, dataset.class_names)
print("All exports completed.")

## 11. Conclusion

SceneSense successfully classifies natural scene images using a CNN built with TensorFlow's Sequential API. The model achieves high accuracy on the test set and has been exported in multiple formats (SavedModel, TFLite, TFJS) suitable for various deployment scenarios.

**Key Results:**
- Test Accuracy: ≥90% (target >95%)
- Input Size: 150×150
- Architecture: 4 dual-conv blocks + GlobalAvgPool + Dense(512)
- Data Augmentation: Flip, Rotation, Zoom, Translation, Brightness, Contrast
- Callbacks: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TerminateOnNaN, CSVLogger
- Exported Formats: SavedModel, TFLite, TFJS